![image.png](https://i.imgur.com/a3uAqnb.png)

# Noise2Noise: Training a Denoiser Without Clean Images

- **Dataset**: [BSDS300](https://huggingface.co/datasets/aseeransari/BSDS) photos for training and validation and [Kodak](https://huggingface.co/datasets/danjacobellis/kodak) photos for testing, both from Hugging Face; the noise is added in the notebook
- **Task**: removing Gaussian noise from colour photos
- **Model**: the U-Net of the Noise2Noise paper (about 1.0M parameters), trained from scratch with clean targets and again with noisy targets only
- **Runtime**: Colab T4 GPU, about 14 minutes of training

In this lab, we will:

- simulate two independent noisy captures of every training photo
- train the Noise2Noise U-Net once with clean targets and once with the second capture as the target
- check numerically that the noisy-target loss is the clean-target loss plus the noise variance
- compare both denoisers with PSNR and SSIM on the Kodak photos

## What is Noise2Noise?

A denoiser $f_\theta$ maps a noisy image $y = x + n$ to an estimate $\hat{x} = f_\theta(y)$ of the clean image $x$. It is usually trained with the mean squared error (MSE) against clean images, which are expensive: a noise-free photo needs a long exposure. Two quick noisy captures of the same scene are cheap.

Noise2Noise trains on such pairs, $y_1 = x + n_1$ and $y_2 = x + n_2$. If $n_2$ has zero mean whatever $x$ and $n_1$ are, as independent zero-mean noise does, then

$$\mathbb{E}\,\|f_\theta(y_1) - y_2\|^2 = \mathbb{E}\,\|f_\theta(y_1) - x\|^2 + \mathbb{E}\,\|n_2\|^2$$

Where:
- $x$ is the clean image, never seen in training; $y_1$ and $y_2$ are the input and target captures, with noise $n_1$ and $n_2$
- $f_\theta$ is the denoiser with weights $\theta$, and $\mathbb{E}$ averages over images and noise

Writing $f_\theta(y_1) - y_2 = (f_\theta(y_1) - x) - n_2$ and expanding the square leaves a cross term $-2\,\mathbb{E}[(f_\theta(y_1) - x)^\top n_2]$, which that condition makes zero, because $f_\theta(y_1) - x$ depends only on $x$ and $n_1$. The last term does not depend on $\theta$, so both losses have the same minimiser; the noisy targets only add variance, to the gradients and, with finite data, to the result. The paper reports 32.50 dB with clean targets and 32.48 dB with noisy ones on the Kodak photos with Gaussian noise of standard deviation $\sigma = 25$ (Table 1, for a 30-layer network trained on 50,000 ImageNet photos).

Paper: [Lehtinen et al., 2018. Noise2Noise: Learning Image Restoration without Clean Data](https://arxiv.org/abs/1803.04189v3)

## Setup

In Google Colab, select **Runtime > Change runtime type > T4 GPU**, then run the cells from top to bottom; Colab has every package the lab needs. The paper's noise level $\sigma = 25$ (of 255) becomes $25/255$ for pixels in $[0, 1]$.

In [ ]:
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from datasets import load_dataset
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

In [ ]:
SEED = 42
N_TRAIN = 200           # BSDS300 training photos (the split has 200)
N_VAL = 100             # BSDS300 test photos, used to track progress after every epoch (the split has 100)
N_TEST = 24             # Kodak photos for the final test (the suite has 24)
NOISE_SIGMA = 25 / 255  # standard deviation of the Gaussian noise, for pixels in [0, 1]
PATCH_SIZE = 128        # side of the random training crops
N_TRAIN_CROPS = 3_200   # crops per epoch: 16 from each training photo
BATCH_SIZE = 16
EPOCHS = 30
LR = 1e-3
NUM_WORKERS = 2         # DataLoader worker processes; a Colab runtime has 2 CPU cores

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__} | device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)} "
          f"({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)")
else:
    print("No GPU found, so training will be slow. In Colab: Runtime > Change runtime type > T4 GPU.")

# Mixed precision: float16 on a T4, which has no bfloat16 units, with a gradient scaler against
# underflow; bfloat16 on an A100 or newer, which needs no scaler; off on the CPU.
USE_AMP = DEVICE == "cuda"
AMP_DTYPE = torch.bfloat16 if USE_AMP and torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
# The training crops always have the same size, so cuDNN can time its convolution algorithms once
# and keep the fastest.
torch.backends.cudnn.benchmark = True

## 1. Dataset

- **Source**: [BSDS300](https://huggingface.co/datasets/aseeransari/BSDS), the 300 colour photos (481x321 or 321x481) of the Berkeley Segmentation Dataset (Martin et al., ICCV 2001), identical to the original release; and the [Kodak image suite](https://huggingface.co/datasets/danjacobellis/kodak), 24 photos of 768x512 or 512x768 stored without lossy compression
- **Licence**: not stated on either card; the [BSDS300 authors](https://www2.eecs.berkeley.edu/Research/Projects/CS/vision/bsds/) offer it "for non-commercial research and educational purposes", and the [Kodak host](https://r0k.us/graphics/kodak/) understands the suite to be released "for unrestricted usage"
- **What we use**: BSDS300's 200 training photos for training and its 100 test photos to track progress; the 24 Kodak photos once, for the final test

Every photo becomes a float tensor in $[0, 1]$; tall BSDS300 photos are turned by 90 degrees so each split stacks into one tensor. The U-Net halves the resolution five times, so validation photos are cropped to 320x480, whose sides divide by $2^5 = 32$.

In [ ]:
# Both repositories are pinned to the revision this lab was tested with.
bsds = load_dataset("aseeransari/BSDS", revision="e4f3281047cbbf0202b0193bd80ce6b7e9a66839")
kodak = load_dataset("danjacobellis/kodak", revision="8ad2123f7cc141390636df3632dda7c77a7914b1")["validation"]
print(bsds)


def to_tensor(image):
    """PIL image -> float tensor (3, H, W) with values in [0, 1]."""
    array = np.asarray(image.convert("RGB"), dtype=np.float32) / 255.0
    return torch.from_numpy(array).permute(2, 0, 1)


def make_wide(tensor):
    return torch.rot90(tensor, 1, dims=(1, 2)) if tensor.shape[1] > tensor.shape[2] else tensor


train_rows = bsds["train"].select(range(min(N_TRAIN, len(bsds["train"]))))
val_rows = bsds["test"].select(range(min(N_VAL, len(bsds["test"]))))
test_rows = kodak.select(range(min(N_TEST, len(kodak))))

train_clean = torch.stack([make_wide(to_tensor(image)) for image in train_rows["image"]])
val_clean = torch.stack([make_wide(to_tensor(image))[:, :320, :480] for image in val_rows["image"]])
test_clean = [to_tensor(image) for image in test_rows["image"]]
test_names = [path.split("/")[-1] for path in test_rows["path"]]   # kodim01.png, ...

print("training photos:  ", tuple(train_clean.shape))   # (N_TRAIN, 3, 321, 481)
print("validation photos:", tuple(val_clean.shape))     # (N_VAL, 3, 320, 480)
print("test photos:      ", len(test_clean), "of shapes", sorted({tuple(t.shape) for t in test_clean}))

### Two noisy captures of every photo

Each capture is the clean photo plus independent Gaussian noise with standard deviation $\sigma$, drawn once before training, like two real shots from a tripod. The captures are not clipped to $[0, 1]$: clipping would cut off part of the noise in dark and bright regions and move its mean away from zero. Validation and test photos get fixed noise too; validation photos also get a second capture.

In [ ]:
noise_generator = torch.Generator().manual_seed(SEED)


def add_noise(clean):
    """One noisy capture: the clean image plus fresh zero-mean Gaussian noise, not clipped."""
    return clean + NOISE_SIGMA * torch.randn(clean.shape, generator=noise_generator)


train_capture_1 = add_noise(train_clean)
train_capture_2 = add_noise(train_clean)
val_noisy = add_noise(val_clean)
val_noisy_2 = add_noise(val_clean)
test_noisy = [add_noise(clean) for clean in test_clean]

n1 = train_capture_1 - train_clean
n2 = train_capture_2 - train_clean
correlation = ((n1 - n1.mean()) * (n2 - n2.mean())).mean() / (n1.std() * n2.std())
print(f"noise std: capture 1 {n1.std():.4f}, capture 2 {n2.std():.4f} (sigma = {NOISE_SIGMA:.4f})")
print(f"noise mean: capture 1 {n1.mean():+.5f}, capture 2 {n2.mean():+.5f}")
print(f"correlation between the two noises: {correlation:+.4f}")
del n1, n2

One training photo, its two captures, and their difference, which is pure noise.

In [ ]:
def show(tensor):
    """(3, H, W) tensor -> (H, W, 3) array clipped to [0, 1] for imshow."""
    return tensor.clamp(0, 1).permute(1, 2, 0).numpy()


k = 3
difference = (train_capture_1[k] - train_capture_2[k]) / (6 * NOISE_SIGMA) + 0.5   # mid-grey is zero
panels = [(train_clean[k], "clean photo (never used by Noise2Noise)"), (train_capture_1[k], "capture 1"),
          (train_capture_2[k], "capture 2"), (difference, "capture 1 - capture 2")]
fig, axes = plt.subplots(1, 4, figsize=(18, 3.6))
for ax, (image, title) in zip(axes, panels):
    ax.imshow(show(image))
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

### Crops for training

Each example is a random 128x128 crop, cut at the same place from both captures and the clean photo, with a random choice of input capture, rotation by a multiple of 90 degrees and flip. The clean crop is the target of the clean-target run; the Noise2Noise run only *measures* its error against it.

In [ ]:
class PairedCrops(Dataset):
    """Aligned random crops of (input capture, other capture, clean photo)."""

    def __init__(self, clean, capture_1, capture_2, n_crops, patch_size):
        self.clean, self.capture_1, self.capture_2 = clean, capture_1, capture_2
        self.n_crops, self.patch_size = n_crops, patch_size

    def __len__(self):
        return self.n_crops

    def __getitem__(self, i):
        k = i % len(self.clean)                         # cycle through the training photos
        _, h, w = self.clean[k].shape
        # torch's random generator, which PyTorch reseeds in every worker from the loader's generator
        top = int(torch.randint(0, h - self.patch_size + 1, ()))
        left = int(torch.randint(0, w - self.patch_size + 1, ()))
        window = (slice(None), slice(top, top + self.patch_size), slice(left, left + self.patch_size))
        crops = [self.capture_1[k][window], self.capture_2[k][window], self.clean[k][window]]
        if torch.rand(()) < 0.5:                        # either capture can be the input
            crops[0], crops[1] = crops[1], crops[0]
        turns = int(torch.randint(0, 4, ()))
        crops = [torch.rot90(c, turns, dims=(1, 2)) for c in crops]
        if torch.rand(()) < 0.5:
            crops = [c.flip(-1) for c in crops]
        return tuple(crops)                             # input, noisy target, clean; each (3, P, P)


def make_train_loader():
    """A loader over the current captures, with its own seeded generator, so that every run draws
    the same crops in the same order."""
    train_set = PairedCrops(train_clean, train_capture_1, train_capture_2, N_TRAIN_CROPS, PATCH_SIZE)
    return DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
                      num_workers=NUM_WORKERS, pin_memory=DEVICE == "cuda",
                      generator=torch.Generator().manual_seed(SEED))


noisy_input, noisy_target, clean_crop = next(iter(make_train_loader()))
print(noisy_input.shape, noisy_target.shape, clean_crop.shape)   # each (BATCH_SIZE, 3, 128, 128)

## 2. Model

The U-Net of the Noise2Noise paper (appendix, Table 2) has an encoder that halves the resolution five times with max-pooling, so its deepest features see a wide neighbourhood of each pixel, and a decoder that doubles it back with nearest-neighbour upsampling. Skip connections concatenate each encoder resolution with the decoder features of the same size, so fine detail bypasses the bottleneck. The encoder has 48 channels and the decoder 96, narrowing to 64 and 32 before a linear output; every convolution but the last is followed by a leaky ReLU with slope 0.1, and there is no batch normalisation.

In [ ]:
def conv3x3(in_channels, out_channels):
    return nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)


class Noise2NoiseUNet(nn.Module):
    """The U-Net of Lehtinen et al. (2018), appendix Table 2."""

    def __init__(self, channels=3):
        super().__init__()
        self.enc0, self.enc1 = conv3x3(channels, 48), conv3x3(48, 48)
        self.enc2, self.enc3, self.enc4, self.enc5, self.enc6 = (conv3x3(48, 48) for _ in range(5))
        self.dec5a, self.dec5b = conv3x3(96, 96), conv3x3(96, 96)
        self.dec4a, self.dec4b = conv3x3(144, 96), conv3x3(96, 96)
        self.dec3a, self.dec3b = conv3x3(144, 96), conv3x3(96, 96)
        self.dec2a, self.dec2b = conv3x3(144, 96), conv3x3(96, 96)
        self.dec1a, self.dec1b, self.dec1c = conv3x3(96 + channels, 64), conv3x3(64, 32), conv3x3(32, channels)
        for module in self.modules():                  # He initialisation, as in the paper
            if isinstance(module, nn.Conv2d):
                nn.init.kaiming_normal_(module.weight, a=0.1, nonlinearity="leaky_relu")
                nn.init.zeros_(module.bias)
        nn.init.kaiming_normal_(self.dec1c.weight, nonlinearity="linear")   # the output layer has no activation

    def forward(self, noisy):                          # noisy: (B, 3, H, W), H and W divisible by 32
        act = lambda t: F.leaky_relu(t, 0.1)
        up = lambda t: F.interpolate(t, scale_factor=2, mode="nearest")
        x0 = noisy - 0.5                               # pixels to [-0.5, 0.5]
        p1 = F.max_pool2d(act(self.enc1(act(self.enc0(x0)))), 2)   # (B, 48, H/2, W/2)
        p2 = F.max_pool2d(act(self.enc2(p1)), 2)       # (B, 48, H/4, W/4)
        p3 = F.max_pool2d(act(self.enc3(p2)), 2)       # (B, 48, H/8, W/8)
        p4 = F.max_pool2d(act(self.enc4(p3)), 2)       # (B, 48, H/16, W/16)
        p5 = F.max_pool2d(act(self.enc5(p4)), 2)       # (B, 48, H/32, W/32)
        x = act(self.enc6(p5))                         # (B, 48, H/32, W/32)
        x = torch.cat([up(x), p4], dim=1)              # (B, 96, H/16, W/16)
        x = act(self.dec5b(act(self.dec5a(x))))        # (B, 96, H/16, W/16)
        x = torch.cat([up(x), p3], dim=1)              # (B, 144, H/8, W/8)
        x = act(self.dec4b(act(self.dec4a(x))))        # (B, 96, H/8, W/8)
        x = torch.cat([up(x), p2], dim=1)              # (B, 144, H/4, W/4)
        x = act(self.dec3b(act(self.dec3a(x))))        # (B, 96, H/4, W/4)
        x = torch.cat([up(x), p1], dim=1)              # (B, 144, H/2, W/2)
        x = act(self.dec2b(act(self.dec2a(x))))        # (B, 96, H/2, W/2)
        x = torch.cat([up(x), x0], dim=1)              # (B, 99, H, W): the input itself is the last skip
        x = act(self.dec1b(act(self.dec1a(x))))        # (B, 32, H, W)
        return self.dec1c(x) + 0.5                     # (B, 3, H, W): linear output, back to [0, 1]


model = Noise2NoiseUNet().to(DEVICE)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
with torch.no_grad():
    print(model(noisy_input.to(DEVICE)).shape)          # (BATCH_SIZE, 3, 128, 128)
del model

## 3. Measuring Quality: PSNR and SSIM

The **peak signal-to-noise ratio** expresses the MSE between an estimate $\hat{x}$ and the clean image $x$ in decibels, with peak value $L = 1$ and $N$ pixel values; halving the MSE adds about 3 dB:

$$\mathrm{PSNR} = 10 \log_{10} \frac{L^2}{\mathrm{MSE}}, \qquad \mathrm{MSE} = \frac{1}{N}\sum_{i=1}^{N} (x_i - \hat{x}_i)^2$$

The **structural similarity** (SSIM) of [Wang et al., 2004](https://doi.org/10.1109/TIP.2003.819861) compares local means $\mu$, variances $\sigma^2$ and the covariance $\sigma_{x\hat{x}}$ (local statistics here, not the noise level) in 11x11 Gaussian windows with standard deviation 1.5; we average it over windows and colour channels, and 1 means identical:

$$\mathrm{SSIM}(x,\hat{x}) = \frac{(2\mu_x\mu_{\hat{x}} + C_1)(2\sigma_{x\hat{x}} + C_2)}{(\mu_x^2+\mu_{\hat{x}}^2+C_1)(\sigma_x^2+\sigma_{\hat{x}}^2+C_2)}, \qquad C_1 = (0.01L)^2,\ C_2 = (0.03L)^2$$

Estimates are clipped to $[0, 1]$ before they are measured.

In [ ]:
def psnr(estimate, clean):
    """PSNR in dB of each image in a batch; both (B, 3, H, W) with values in [0, 1]."""
    mse = ((estimate.clamp(0, 1) - clean) ** 2).flatten(1).mean(dim=1)
    return 10 * torch.log10(1.0 / mse)


def gaussian_window(size=11, std=1.5):
    coords = torch.arange(size, dtype=torch.float32) - (size - 1) / 2
    g = torch.exp(-coords ** 2 / (2 * std ** 2))
    g = g / g.sum()
    return g[:, None] * g[None, :]                     # (size, size), sums to 1


def ssim(estimate, clean, peak=1.0):
    """SSIM of each image in a batch, averaged over windows and colour channels."""
    estimate = estimate.clamp(0, 1)
    channels = clean.shape[1]
    window = gaussian_window().to(clean)[None, None].repeat(channels, 1, 1, 1)   # (3, 1, 11, 11)
    blur = lambda t: F.conv2d(t, window, groups=channels)                         # local means, per channel
    mu_x, mu_e = blur(clean), blur(estimate)
    var_x = blur(clean * clean) - mu_x ** 2
    var_e = blur(estimate * estimate) - mu_e ** 2
    cov = blur(clean * estimate) - mu_x * mu_e
    c1, c2 = (0.01 * peak) ** 2, (0.03 * peak) ** 2
    ssim_map = ((2 * mu_x * mu_e + c1) * (2 * cov + c2)) / ((mu_x ** 2 + mu_e ** 2 + c1) * (var_x + var_e + c2))
    return ssim_map.flatten(1).mean(dim=1)

The simplest "denoiser" returns its input; its Kodak scores are the baseline to beat.

In [ ]:
noisy_psnr = torch.cat([psnr(noisy[None], clean[None]) for noisy, clean in zip(test_noisy, test_clean)])
noisy_ssim = torch.cat([ssim(noisy[None], clean[None]) for noisy, clean in zip(test_noisy, test_clean)])
print(f"Noisy input on Kodak: PSNR {noisy_psnr.mean():.2f} dB, SSIM {noisy_ssim.mean():.4f}")

## 4. Training

Two fresh U-Nets train with the MSE loss, Adam and a cosine learning-rate decay, from the same seed (same initial weights, same crops in the same order) for the same number of steps. Only the target differs:

- **Noise2Clean**, the usual recipe: input $y_1$, target the clean crop $x$
- **Noise2Noise**: input $y_1$, target the other capture $y_2$

Both runs also record their error against the clean crops, which Noise2Noise could not measure in practice. After each epoch, we measure the validation PSNR and the MSE against the clean photos and their second captures.

In [ ]:
def denoise(model, noisy, batch_size=4):
    """Runs the model over a (N, 3, H, W) tensor in small batches; returns float32 on the CPU."""
    model.eval()
    outputs = []
    with torch.no_grad():
        for i in range(0, len(noisy), batch_size):
            batch = noisy[i:i + batch_size].to(DEVICE)
            with torch.autocast(device_type=DEVICE, dtype=AMP_DTYPE, enabled=USE_AMP):
                outputs.append(model(batch).float().cpu())
    return torch.cat(outputs)


def train_one_epoch(model, loader, optimizer, scheduler, scaler, target_kind):
    """One pass over the crops. Returns the mean training loss and the mean MSE against the clean crops."""
    model.train()
    total_loss, total_clean_mse, seen = 0.0, 0.0, 0
    # Pass an iterator, not the loader: given a loader, tqdm.auto calls iter() on it once more for
    # itself, which starts an extra set of worker processes every epoch that is never used or shut down.
    for noisy_input, noisy_target, clean in tqdm(iter(loader), total=len(loader), desc="train", leave=False):
        noisy_input = noisy_input.to(DEVICE, non_blocking=True)
        noisy_target = noisy_target.to(DEVICE, non_blocking=True)
        clean = clean.to(DEVICE, non_blocking=True)
        target = {"clean": clean, "noisy": noisy_target, "input": noisy_input}[target_kind]
        with torch.autocast(device_type=DEVICE, dtype=AMP_DTYPE, enabled=USE_AMP):
            estimate = model(noisy_input)              # (B, 3, P, P)
        loss = F.mse_loss(estimate.float(), target)    # in float32, for precision
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()                  # the scaler is a no-op when disabled
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        with torch.no_grad():
            clean_mse = F.mse_loss(estimate.float(), clean)   # measured only; not part of the loss
        total_loss += loss.item() * len(clean)
        total_clean_mse += clean_mse.item() * len(clean)
        seen += len(clean)
    return total_loss / seen, total_clean_mse / seen


@torch.no_grad()
def evaluate(model, noisy, clean, noisy_2):
    """Mean PSNR on held-out photos, and the MSE against the clean photos and against their second captures."""
    estimate = denoise(model, noisy)
    return (psnr(estimate, clean).mean().item(),
            F.mse_loss(estimate, clean).item(),
            F.mse_loss(estimate, noisy_2).item())

One complete run. `target_kind` is `"clean"` for Noise2Clean, `"noisy"` for Noise2Noise, or `"input"` (the input itself, for an exercise).

In [ ]:
def train_denoiser(target_kind):
    """Trains a fresh U-Net for EPOCHS epochs with the chosen target; returns the model and its history."""
    torch.manual_seed(SEED)                            # same initial weights and same crops in every run
    model = Noise2NoiseUNet().to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, betas=(0.9, 0.99))   # the paper's betas
    loader = make_train_loader()
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS * len(loader))
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP and AMP_DTYPE == torch.float16)
    history = {"train_loss": [], "train_clean_mse": [], "val_psnr": [], "val_clean_mse": [], "val_noisy_mse": []}
    print(f"[{target_kind} targets] {len(loader):,} optimizer steps per epoch, {EPOCHS} epochs")
    start = time.time()
    for epoch in range(1, EPOCHS + 1):
        train_loss, train_clean_mse = train_one_epoch(model, loader, optimizer, scheduler, scaler, target_kind)
        val_psnr, val_clean_mse, val_noisy_mse = evaluate(model, val_noisy, val_clean, val_noisy_2)
        for key, value in zip(history, (train_loss, train_clean_mse, val_psnr, val_clean_mse, val_noisy_mse)):
            history[key].append(value)
        print(f"Epoch {epoch}/{EPOCHS}: train loss {train_loss:.5f} (vs clean {train_clean_mse:.5f}) | "
              f"val PSNR {val_psnr:.2f} dB | {time.time() - start:.0f}s elapsed")
    return model, history

First the usual recipe, with clean targets. Expected time: about 7 minutes on a T4.

In [ ]:
n2c_model, n2c_history = train_denoiser("clean")

Then Noise2Noise. Its loss stays high because the target is noisy; watch the error against the clean crops, in brackets. Expected time: about 7 minutes on a T4.

In [ ]:
n2n_model, n2n_history = train_denoiser("noisy")

## 5. Results

### Training curves

Left: the training losses (log scale), the Noise2Noise run's error against the clean crops (dashed) and the noise variance $\sigma^2$ (dotted). Right: the validation PSNR.

In [ ]:
epochs = range(1, len(n2c_history["train_loss"]) + 1)
fig, (ax_loss, ax_psnr) = plt.subplots(1, 2, figsize=(13, 4.5))
ax_loss.plot(epochs, n2c_history["train_loss"], marker="o", label="Noise2Clean loss (clean target)")
ax_loss.plot(epochs, n2n_history["train_loss"], marker="o", label="Noise2Noise loss (noisy target)")
ax_loss.plot(epochs, n2n_history["train_clean_mse"], linestyle="--", label="Noise2Noise, error vs clean (not trained on)")
ax_loss.axhline(NOISE_SIGMA ** 2, color="grey", linestyle=":", label=r"noise variance $\sigma^2$")
ax_loss.set(xlabel="epoch", ylabel="MSE", yscale="log", title="Training loss")
ax_loss.legend(fontsize=8)
ax_psnr.plot(epochs, n2c_history["val_psnr"], marker="o", label="Noise2Clean")
ax_psnr.plot(epochs, n2n_history["val_psnr"], marker="o", label="Noise2Noise")
ax_psnr.set(xlabel="epoch", ylabel="PSNR (dB)", title="Validation PSNR (BSDS300 test photos)")
ax_psnr.legend()
plt.tight_layout()
plt.show()

### The loss decomposition on held-out photos

For each model: the MSE against the second validation capture $y_2$, the clean-target MSE, the noise power $\frac{1}{N}\sum n_2^2$, and the cross term $-\frac{2}{N}\sum (f_\theta(y_1) - x)\, n_2$, which should vanish.

In [ ]:
@torch.no_grad()
def loss_decomposition(model):
    estimate = denoise(model, val_noisy)               # f(y1), not clipped
    n2 = val_noisy_2 - val_clean                       # the noise of the second capture
    return {"noisy-target MSE": F.mse_loss(estimate, val_noisy_2).item(),
            "clean-target MSE": F.mse_loss(estimate, val_clean).item(),
            "noise power": (n2 ** 2).mean().item(),
            "cross term": (-2 * (estimate - val_clean) * n2).mean().item()}


decomposition = pd.DataFrame({"Noise2Clean": loss_decomposition(n2c_model),
                              "Noise2Noise": loss_decomposition(n2n_model)}).T
decomposition["clean MSE + noise power"] = decomposition["clean-target MSE"] + decomposition["noise power"]
print(f"sigma^2 = {NOISE_SIGMA ** 2:.6f}")
print(decomposition.to_string(float_format=lambda v: f"{v:.6f}"))

### The final test on Kodak

Both models denoise the 24 Kodak photos at full resolution; each scatter point is one photo.

In [ ]:
@torch.no_grad()
def test_on_kodak(model):
    """Denoised Kodak photos, and the PSNR and SSIM of each."""
    estimates = [denoise(model, noisy[None])[0] for noisy in test_noisy]
    psnrs = torch.cat([psnr(e[None], c[None]) for e, c in zip(estimates, test_clean)])
    ssims = torch.cat([ssim(e[None], c[None]) for e, c in zip(estimates, test_clean)])
    return estimates, psnrs, ssims


n2c_estimates, n2c_psnr, n2c_ssim = test_on_kodak(n2c_model)
n2n_estimates, n2n_psnr, n2n_ssim = test_on_kodak(n2n_model)

results = pd.DataFrame({
    "PSNR (dB)": [noisy_psnr.mean().item(), n2c_psnr.mean().item(), n2n_psnr.mean().item()],
    "SSIM": [noisy_ssim.mean().item(), n2c_ssim.mean().item(), n2n_ssim.mean().item()],
}, index=["noisy input", "Noise2Clean (clean targets)", "Noise2Noise (noisy targets only)"])
print(results.round(4).to_string())
gap = n2n_psnr - n2c_psnr
print(f"\nNoise2Noise minus Noise2Clean, per photo: mean {gap.mean():+.3f} dB, range {gap.min():+.3f} to {gap.max():+.3f} dB")

low, high = min(n2c_psnr.min(), n2n_psnr.min()).item() - 0.5, max(n2c_psnr.max(), n2n_psnr.max()).item() + 0.5
plt.figure(figsize=(5, 5))
plt.scatter(n2c_psnr, n2n_psnr)
plt.plot([low, high], [low, high], color="grey", linestyle="--", label="equal PSNR")
plt.xlabel("Noise2Clean PSNR (dB)")
plt.ylabel("Noise2Noise PSNR (dB)")
plt.title("Kodak photos, one point each")
plt.legend()
plt.axis("equal")
plt.show()

Close-ups of three photos; each title gives the PSNR of the whole photo.

In [ ]:
close_ups = [("kodim08.png", 180, 300), ("kodim19.png", 400, 120), ("kodim23.png", 120, 330)]   # (photo, top, left)
close_ups = [c for c in close_ups if c[0] in test_names] or [(test_names[0], 0, 0)]   # when N_TEST is small
fig, axes = plt.subplots(len(close_ups), 4, figsize=(14, 3.6 * len(close_ups)), squeeze=False)
for row, (name, top, left) in zip(axes, close_ups):
    i = test_names.index(name)
    window = (slice(None), slice(top, top + 192), slice(left, left + 192))
    panels = [(test_clean[i], f"{name}: clean"), (test_noisy[i], f"noisy, {noisy_psnr[i]:.2f} dB"),
              (n2c_estimates[i], f"Noise2Clean, {n2c_psnr[i]:.2f} dB"),
              (n2n_estimates[i], f"Noise2Noise, {n2n_psnr[i]:.2f} dB")]
    for ax, (image, title) in zip(row, panels):
        ax.imshow(show(image[window]))
        ax.set_title(title)
        ax.axis("off")
plt.tight_layout()
plt.show()

**What to notice**

The Noise2Noise loss stays above the Noise2Clean loss, near the noise variance: in the paper's words, every training example "asks for the impossible", since no network can predict one noise sample from another. Yet its error against the clean crops follows the Noise2Clean loss, and the validation PSNR curves almost coincide. The decomposition shows why: for both models, the noisy-target MSE equals the clean-target MSE plus the noise power, with a cross term near zero.

On Kodak, both models beat the noisy input by a wide margin. The points hug the diagonal, with Noise2Noise a little below it on most photos: the noisy targets add variance, which costs a little at a fixed budget. Both outputs are smoother than the clean photos in fine texture: an MSE denoiser approximates the probability-weighted average of the clean images that could have produced its input.

### Save and reload

We save the Noise2Noise weights, reload them into a fresh U-Net and check the Kodak score.

In [ ]:
torch.save(n2n_model.state_dict(), "noise2noise_unet.pth")

reloaded = Noise2NoiseUNet().to(DEVICE)
reloaded.load_state_dict(torch.load("noise2noise_unet.pth", map_location=DEVICE))
_, reloaded_psnr, _ = test_on_kodak(reloaded)
print(f"Reloaded Noise2Noise model on Kodak: {reloaded_psnr.mean():.2f} dB (before saving: {n2n_psnr.mean():.2f} dB)")

## Summary

- With an MSE loss, a noisy target $y_2 = x + n_2$ only adds the constant $\mathbb{E}\|n_2\|^2$ when $n_2$ has zero mean whatever the scene and the input, so the minimiser is unchanged.
- Two noisy captures per scene were enough: without seeing a clean image, the Noise2Noise U-Net came close to the clean-target U-Net on Kodak.
- The MSE against held-out second captures differs from the clean MSE by the same constant, so it can compare models when no clean images exist.
- Noise2Noise needs no noise model and no clean reference, but with the MSE loss the target noise must stay zero-mean: clipping the captures, or reusing the input's own noise as the target, breaks it.

## Exercises

1. **Copy the input as the target.** Set `EPOCHS = 5`, run `train_denoiser("input")` and score it with `test_on_kodak`. Why does it return its input, and what happens to the cross term when $n_2 = n_1$?
2. **Clip the captures.** Add `.clamp(0, 1)` to what `add_noise` returns and rerun every cell from there on. Compare very dark and very bright regions of both models' outputs with the clean photos.
3. **Signal-dependent noise.** In `add_noise`, replace `NOISE_SIGMA` by `torch.sqrt(0.0004 + 0.01 * clean)`, for a noise variance $0.0004 + 0.01\,x$ that grows with brightness as in a camera sensor, and rerun every cell from there on. Does Noise2Noise still match Noise2Clean?
4. **Validate without clean data.** Plot `val_noisy_mse` and `val_clean_mse` of `n2n_history` by epoch. What is the gap between them, and how could the noisy curve pick a model without clean images?

## Further Reading

- [Lehtinen et al., 2018. Noise2Noise: Learning Image Restoration without Clean Data](https://arxiv.org/abs/1803.04189v3)
- [Krull et al., 2019. Noise2Void - Learning Denoising from Single Noisy Images](https://arxiv.org/abs/1811.10980)
- [Batson and Royer, 2019. Noise2Self: Blind Denoising by Self-Supervision](https://arxiv.org/abs/1901.11365)
- [Ronneberger et al., 2015. U-Net: Convolutional Networks for Biomedical Image Segmentation](https://arxiv.org/abs/1505.04597)

### Contributed by: Ali Habibullah